In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [10]:
function union_bounding_spheres(sphere1::RayTracing.Sphere, sphere2::RayTracing.Sphere)::RayTracing.Sphere
    center1 = sphere1.core.world_to_object(RayTracing.Pnt3(0,0,0))
    center2 = sphere2.core.world_to_object(RayTracing.Pnt3(0,0,0))

    # Get the vector between the centers
    center_vector = center2 - center1
    center_distance = RayTracing.norm(center_vector)
    
    # If one sphere contains the other, return the larger one
    if center_distance + sphere2.radius <= sphere1.radius
        # sphere1 completely contains sphere2
        return sphere1
    elseif center_distance + sphere1.radius <= sphere2.radius
        # sphere2 completely contains sphere1
        return sphere2
    end
    
    # Otherwise, create a new sphere that encloses both
    # The new center is along the line between the centers
    # weighted by the radii
    new_center = center1 + center_vector * 0.5
    
    # The new radius must reach the furthest point of either sphere
    new_radius = (center_distance + sphere1.radius + sphere2.radius) * 0.5
    
    return RayTracing.Sphere(new_center, new_radius)
end

union_bounding_spheres (generic function with 1 method)

In [73]:
################
### ABSTRACT ###
################

# Base abstract type for all SDF components
abstract type ImplicitSurface end

# Primitive SDF shapes inherit from this
abstract type SDFPrimitive <: ImplicitSurface end

# Operations (union, intersection, etc.) inherit from this
abstract type SDFOperation <: ImplicitSurface end

##################
### OPERATIONS ###
##################

# Define specific operation types
struct SDFUnion <: SDFOperation
    k::Float64  # Smoothing parameter
    left::ImplicitSurface
    right::ImplicitSurface
    bounding_sphere::RayTracing.Sphere

    function SDFUnion(k::Float64, left::ImplicitSurface, right::ImplicitSurface)
        return new(k, left, right, union_bounding_spheres(left.bounding_sphere, right.bounding_sphere))
    end
end

##################
### PRIMITIVES ###
##################

struct SDFSphere <: SDFPrimitive
    radius::Float64
    core::RayTracing.ShapeCore
    bounding_sphere::RayTracing.Sphere
end

struct SDFBox <: SDFPrimitive
    half_extents::RayTracing.Pnt3  # half-width, half-height, half-depth
    core::RayTracing.ShapeCore
    bounding_sphere::RayTracing.Sphere
end

###################
### EVALUTATION ###
###################

# Primitive evaluations
function evaluate(shape::SDFSphere, p::RayTracing.Pnt3)::Float64
    # Transform point to object space
    local_p = shape.core.world_to_object(p)
    # Sphere SDF: length(p) - radius
    return RayTracing.norm(local_p) - shape.radius
end

function evaluate(shape::SDFBox, p::RayTracing.Pnt3)::Float64
    # Transform point to object space
    local_p = shape.core.world_to_object(p)
    # Box SDF implementation
    q = abs.(local_p) .- shape.half_extents
    return RayTracing.norm(max.(q, 0.0)) + min(maximum(q), 0.0)
end

# Operation evaluations
function evaluate(op::SDFUnion, p::RayTracing.Pnt3)::Float64
    a = evaluate(op.left, p) 
    b = evaluate(op.right, p)
    
    # Smooth union formula
    if op.k > 0
        h = clamp(0.5 + 0.5 * (b - a) / op.k, 0.0, 1.0)
        return min(b, a, h) - op.k * h * (1.0 - h)
    else
        # Regular union
        return min(a, b)
    end
end

function evaluate(op::SDFIntersection, p::RayTracing.Pnt3)::Float64
    a = evaluate(op.left, p)
    b = evaluate(op.right, p)
    
    # Smooth intersection formula
    if op.k > 0
        h = clamp(0.5 - 0.5 * (b - a) / op.k, 0.0, 1.0)
        return mix(b, a, h) + op.k * h * (1.0 - h)
    else
        # Regular intersection
        return max(a, b)
    end
end

#################
### INTERFACE ###
#################

# Main evaluation function - dispatches to specialized methods
function f(element::ImplicitSurface, p::RayTracing.Pnt3)::Float64
    return evaluate(element, p)
end

# Compatibility with ray evaluation
function f(element::ImplicitSurface, t::Float64, r::RayTracing.AbstractRay)::Float64
    return f(element, RayTracing.at(r, t))
end

# Create two primitive shapes
sphere = SDFSphere(
    1.0, 
    RayTracing.ShapeCore(), 
    RayTracing.Sphere(RayTracing.Pnt3(0,0,0), 1.0 * 1.1)
)
box = SDFBox(RayTracing.Pnt3(1.0, 1.0, 1.0), RayTracing.ShapeCore(), RayTracing.Sphere(RayTracing.Pnt3(0,0,0), 3.0 * 1.1))

# Create a union of the two shapes
union_shape = SDFUnion(0.0, sphere, box)

# Evaluate the SDF at a point
a = f(sphere, RayTracing.Pnt3(10.0, 7.0, 5.0))
b = f(box, RayTracing.Pnt3(10.0, 7.0, 5.0))

11.532562594670797

# why no smooth union working?

In [76]:
k = 0.1

0.1

In [77]:
k *= 6.0
h = max( k-abs(a-b), 0.0 )/k
return min(a,b) - h*h*h*k*(1.0/6.0)

11.532562594670797

In [75]:
min(b, a, h)

0.0

In [ ]:
- k * h * (1.0 - h)

In [48]:
o = RayTracing.Pnt3(10, 7, 5)

r = RayTracing.Ray(
    o,
    RayTracing.normalize(RayTracing.Vec3(RayTracing.Pnt3(0,0,0) - o)),
    0.0,
    typemax(Float64)
)

[10.0, 7.0, 5.0], [-0.7580980435789033, -0.5306686305052324, -0.37904902178945166], 0.0, Inf, false

In [49]:
function intersect_t(s::Union{SDFUnion, SDFPrimitive}, r::RayTracing.AbstractRay)::Float64
    # set up anonymous function for solver
    tmp_solve = (x -> f(s, x, r))

    # intersect bounding sphere
    check, t0, t1 = RayTracing.intersect_simple(s.bounding_sphere, r)

    # doesn't intersect sphere, NEXT
    if !check
        return -1.0
    else
        @info "Implicit Surface Bounding Sphere Intersection: $(RayTracing.at(r, t0)), $(t0), $(t1)"
    end

    # TODO some checks t0 & t1 aren't negative?

    # skipping solve if start point is zero
    if tmp_solve(t0) == 0.0
        solutions = [t0]
    elseif tmp_solve(t1) == 0.0
        solutions = [t1]
    else
        @info "Entering Solve: $r"
        solutions = RayTracing.find_zeros(tmp_solve, t0, t1) # HACKY
        @info "Exiting Solve: $solutions"
    end

    @info "ImplicitSurfaceIntersectionTest: ray: $(r), solutions: $(solutions), bounding_sphere bounds: ($(t0/1.1), $(t1*1.1))"

    if length(solutions) == 0
        @info "Length of solutions is zero"
        return -1.0
    end

    # find intersection time
    _, idx = findmin(abs.(solutions))
    t = solutions[idx]

    if t > r.tMax
        @info "t out of bounds for the ray"
        return -1.0
    end

    return t
end

intersect_t (generic function with 1 method)

In [59]:
t = intersect_t(sphere, r)

┌ Info: Implicit Surface Bounding Sphere Intersection: [0.8339078479367963, 0.583735493555757, 0.41695392396839814], 12.090905958272916, 14.290905958272923
└ @ Main /Users/johnmyslinski/Documents/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl:12
┌ Info: Entering Solve: [10.0, 7.0, 5.0], [-0.7580980435789033, -0.5306686305052324, -0.37904902178945166], 0.0, Inf, false
└ @ Main /Users/johnmyslinski/Documents/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl:23
┌ Info: Exiting Solve: [12.19090595827292, 14.19090595827292]
└ @ Main /Users/johnmyslinski/Documents/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl:25
┌ Info: ImplicitSurfaceIntersectionTest: ray: [10.0, 7.0, 5.0], [-0.7580980435789033, -0.5306686305052324, -0.37904902178945166], 0.0, Inf, false, solutions: [12.19090595827292, 14.19090595827292], bounding_sphere bounds: (10.991732689339013, 15.719996554100216)
└ @ Main /U

12.19090595827292

In [60]:
RayTracing.at(r, t)

3-element Main.RayTracing.Pnt3 with indices SOneTo(3):
 0.7580980435789044
 0.5306686305052324
 0.3790490217894522

In [67]:
distance = f(sphere, RayTracing.at(r, t))

8.881784197001252e-16

# Quad Debugging

In [2]:
struct SDFQuad <: RayTracing.SDFPrimitive
    a::RayTracing.Vec3
    b::RayTracing.Vec3
    c::RayTracing.Vec3
    d::RayTracing.Vec3
    core::RayTracing.ShapeCore
    bounding_sphere::RayTracing.Sphere

    function SDFQuad(a::RayTracing.Vec3, b::RayTracing.Vec3, c::RayTracing.Vec3, d::RayTracing.Vec3, core::RayTracing.ShapeCore, bounding_sphere::RayTracing.Sphere)::SDFQuad
        ba = b - a
        ad = a - d
        nor = RayTracing.cross(ba, ad)
        plane_tolerance = 1e-10
        nor_normalized = nor / RayTracing.norm(nor)
    
        # let's make sure your vertices aren't ordered incorrectly
        @assert abs(RayTracing.dot(nor_normalized, b - a)) < plane_tolerance "Vertices are not coplanar - check vertex ordering"
        @assert abs(RayTracing.dot(nor_normalized, c - a)) < plane_tolerance "Vertices are not coplanar - check vertex ordering"  
        @assert abs(RayTracing.dot(nor_normalized, d - a)) < plane_tolerance "Vertices are not coplanar - check vertex ordering"
        @assert RayTracing.norm(nor) > plane_tolerance "Degenerate quadrilateral - vertices are collinear or coincident"
        return new(a, b, c, d, core, bounding_sphere)
    end
end

In [3]:
quad = SDFQuad(
    RayTracing.Vec3(25, -3, 25),
    RayTracing.Vec3(25, -3, -25),
    RayTracing.Vec3(-25, -3, -25),
    RayTracing.Vec3(-25, -3, 25),
    RayTracing.ShapeCore(),
    RayTracing.Sphere(RayTracing.Pnt3(0,-3,0), 50 * sqrt(2.0))
)

SDFQuad([25.0, -3.0, 25.0], [25.0, -3.0, -25.0], [-25.0, -3.0, -25.0], [-25.0, -3.0, 25.0], Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), false, false), Main.RayTracing.Sphere(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 -3.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 3.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 3.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 -3.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), false, false), 70.71067811865476, -70.71067811865476, 70.71067811865476, 0.0, 3.141592653589793, 6.283185307179586))

In [4]:
function evaluate(shape::SDFQuad, p::RayTracing.Pnt3)::Float64
    ba = shape.b - shape.a
    pa = p       - shape.a
    cb = shape.c - shape.b
    pb = p       - shape.b  
    dc = shape.d - shape.c
    pc = p       - shape.c
    ad = shape.a - shape.d
    pd = p       - shape.d
    
    nor = RayTracing.cross(ba, ad)
    
    nor_normalized = nor / RayTracing.norm(nor)

    dot2(v) = RayTracing.dot(v, v)
    
    inside_test = (sign(RayTracing.dot(RayTracing.cross(ba, nor), pa)) +
                   sign(RayTracing.dot(RayTracing.cross(cb, nor), pb)) + 
                   sign(RayTracing.dot(RayTracing.cross(dc, nor), pc)) +
                   sign(RayTracing.dot(RayTracing.cross(ad, nor), pd)))
    
    if inside_test < 3.0
        dist_ba = dot2(ba * clamp(RayTracing.dot(ba, pa) / dot2(ba), 0.0, 1.0) - pa)
        dist_cb = dot2(cb * clamp(RayTracing.dot(cb, pb) / dot2(cb), 0.0, 1.0) - pb)
        dist_dc = dot2(dc * clamp(RayTracing.dot(dc, pc) / dot2(dc), 0.0, 1.0) - pc)
        dist_ad = dot2(ad * clamp(RayTracing.dot(ad, pd) / dot2(ad), 0.0, 1.0) - pd)
        
        return sqrt(min(dist_ba, dist_cb, dist_dc, dist_ad))
    else
        return sqrt(RayTracing.dot(nor, pa)^2 / dot2(nor))
    end
end

evaluate (generic function with 1 method)

In [12]:
evaluate(quad, RayTracing.Pnt3(27, -3, 1))

2.0

In [ ]:
using LinearAlgebra
"""
    udQuad_debug(p, a, b, c, d; verbose=false)

Debug version of udQuad with detailed output to help diagnose issues.
"""
function udQuad_debug(p::Vector{Float64}, a::Vector{Float64}, b::Vector{Float64}, 
                      c::Vector{Float64}, d::Vector{Float64}; verbose::Bool=false)::Float64
    
    # Edge vectors and point-to-vertex vectors
    ba = b - a; pa = p - a
    cb = c - b; pb = p - b  
    dc = d - c; pc = p - c
    ad = a - d; pd = p - d
    
    # Normal vector to the quad plane
    nor = cross(ba, ad)
    
    if verbose
        println("Edge vectors:")
        println("  ba = ", ba)
        println("  cb = ", cb) 
        println("  dc = ", dc)
        println("  ad = ", ad)
        println("Normal vector: ", nor)
        println("Normal magnitude: ", norm(nor))
    end
    
    # Assert proper vertex ordering (with relaxed tolerance for debugging)
    plane_tolerance = 1e-6  # More lenient for debugging
    
    if norm(nor) < plane_tolerance
        @warn "Degenerate quadrilateral - vertices are collinear or coincident"
        return Inf
    end
    
    nor_normalized = nor / norm(nor)
    
    # Check coplanarity
    dist_b = abs(dot(nor_normalized, b - a))
    dist_c = abs(dot(nor_normalized, c - a))  
    dist_d = abs(dot(nor_normalized, d - a))
    
    if verbose
        println("Coplanarity check:")
        println("  Distance b to plane: ", dist_b)
        println("  Distance c to plane: ", dist_c)
        println("  Distance d to plane: ", dist_d)
    end
    
    if max(dist_b, dist_c, dist_d) > plane_tolerance
        @warn "Vertices may not be coplanar - max deviation: $(max(dist_b, dist_c, dist_d))"
    end
    
    # Helper function for dot product of vector with itself
    dot2(v) = dot(v, v)
    
    # Calculate the inside test components
    test1 = sign(dot(cross(ba, nor), pa))
    test2 = sign(dot(cross(cb, nor), pb))
    test3 = sign(dot(cross(dc, nor), pc))
    test4 = sign(dot(cross(ad, nor), pd))
    
    inside_sum = test1 + test2 + test3 + test4
    
    if verbose
        println("Inside test components:")
        println("  Edge ba test: ", test1)
        println("  Edge cb test: ", test2)
        println("  Edge dc test: ", test3)
        println("  Edge ad test: ", test4)
        println("  Sum: ", inside_sum)
        println("  Inside test: ", inside_sum, " < 4.0 = ", inside_sum < 4.0)
    end
    
    if inside_sum < 4
        # Point projects outside quad - return distance to nearest edge
        
        # Distance to each edge (clamped to edge endpoints)
        t_ba = clamp(dot(ba, pa) / dot2(ba), 0.0, 1.0)
        t_cb = clamp(dot(cb, pb) / dot2(cb), 0.0, 1.0)
        t_dc = clamp(dot(dc, pc) / dot2(dc), 0.0, 1.0)
        t_ad = clamp(dot(ad, pd) / dot2(ad), 0.0, 1.0)
        
        closest_ba = ba * t_ba
        closest_cb = cb * t_cb
        closest_dc = dc * t_dc
        closest_ad = ad * t_ad
        
        dist_ba = dot2(closest_ba - pa)
        dist_cb = dot2(closest_cb - pb)
        dist_dc = dot2(closest_dc - pc)
        dist_ad = dot2(closest_ad - pd)
        
        min_dist = min(dist_ba, dist_cb, dist_dc, dist_ad)
        
        if verbose
            println("Edge distance calculation:")
            println("  Distance to ba: ", sqrt(dist_ba), " (t=", t_ba, ")")
            println("  Distance to cb: ", sqrt(dist_cb), " (t=", t_cb, ")")
            println("  Distance to dc: ", sqrt(dist_dc), " (t=", t_dc, ")")
            println("  Distance to ad: ", sqrt(dist_ad), " (t=", t_ad, ")")
            println("  Minimum: ", sqrt(min_dist))
            println("  → Using EDGE distance")
        end
        
        return sqrt(min_dist)
    else
        # Point projects inside quad - return perpendicular distance to plane
        plane_dist_sq = dot(nor, pa)^2 / dot2(nor)
        
        if verbose
            println("Plane distance calculation:")
            println("  Plane distance: ", sqrt(plane_dist_sq))
            println("  → Using PLANE distance")
        end
        
        return sqrt(plane_dist_sq)
    end
end

# Test function to help diagnose issues
function test_quad_basic()
    println("=== Basic Quad Test ===")
    
    # Simple unit square in xy-plane
    a = [0.0, 0.0, 0.0]
    b = [1.0, 0.0, 0.0]
    c = [1.0, 1.0, 0.0] 
    d = [0.0, 1.0, 0.0]
    
    # Test points
    center = [0.5, 0.5, 0.0]  # Should be on plane
    above = [0.5, 0.5, 1.0]   # Should be distance 1.0
    outside = [2.0, 0.5, 0.0] # Should be distance 1.0 (to edge)
    
    println("Center point (should be ~0):")
    dist1 = udQuad_debug(center, a, b, c, d, verbose=true)
    println("Result: ", dist1)
    println()
    
    println("Above center (should be 1.0):")
    dist2 = udQuad_debug(above, a, b, c, d, verbose=true)
    println("Result: ", dist2)
    println()
    
    println("Outside edge (should be 1.0):")
    dist3 = udQuad_debug(outside, a, b, c, d, verbose=true)
    println("Result: ", dist3)
    println()
end

test_quad_basic (generic function with 1 method)

In [6]:
test_quad_basic()

=== Basic Quad Test ===
Center point (should be ~0):
Edge vectors:
  ba = [1.0, 0.0, 0.0]
  cb = [0.0, 1.0, 0.0]
  dc = [-1.0, 0.0, 0.0]
  ad = [0.0, -1.0, 0.0]
Normal vector: [0.0, 0.0, -1.0]
Normal magnitude: 1.0
Coplanarity check:
  Distance b to plane: 0.0
  Distance c to plane: 0.0
  Distance d to plane: 0.0
Inside test components:
  Edge ba test: 1.0
  Edge cb test: 1.0
  Edge dc test: 1.0
  Edge ad test: 1.0
  Sum: 4.0
  Inside test: 4.0 < 3.0 = false
Plane distance calculation:
  Plane distance: 0.0
  → Using PLANE distance
Result: 0.0

Above center (should be 1.0):
Edge vectors:
  ba = [1.0, 0.0, 0.0]
  cb = [0.0, 1.0, 0.0]
  dc = [-1.0, 0.0, 0.0]
  ad = [0.0, -1.0, 0.0]
Normal vector: [0.0, 0.0, -1.0]
Normal magnitude: 1.0
Coplanarity check:
  Distance b to plane: 0.0
  Distance c to plane: 0.0
  Distance d to plane: 0.0
Inside test components:
  Edge ba test: 1.0
  Edge cb test: 1.0
  Edge dc test: 1.0
  Edge ad test: 1.0
  Sum: 4.0
  Inside test: 4.0 < 3.0 = false
Plane dis